# CC1π systematics

In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp
import ROOT
from ROOT import TMVA
import pickle

from analysis_village.cc1pi.var_configs import *

import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd
import gc

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)


from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks import CutMasks
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

# load dataframes

In [ ]:
#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_bnb_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_5e18_CV.df", keys2load, 100)
mc_bnb_evt_df = mc_bnb_df['cc1pi']
mc_bnb_nu_df = mc_bnb_df['nudf']
mc_bnb_hdr_df = mc_bnb_df['hdr']

#Load data
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_rollingdev_bnblight.df", keys2load, 100)
data_evt_df = data_df['cc1pi']
data_hdr_df = data_df['hdr']

# POT

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')
# BNB data
data_tot_pot = data_hdr_df['pot'].sum()
print("data_tot_pot: %.3e" %(data_tot_pot))
data_evt_df[pot_weight_col] = np.ones(len(data_evt_df))
data_gates = data_hdr_df.nbnbinfo.sum()
print("data tot gates : %.3e" %(data_gates))


# BNB MC
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_evt_df))

In [ ]:
mc_evt_df = perform_truth_matching(mc_bnb_evt_df, mc_bnb_nu_df)

In [ ]:
## total pot
slc_df = (
        mc_evt_df
        .groupby(['__ntuple', 'entry', 'rec.slc..index'])
        .first()
    )
slc_df = slc_df.sort_index()

# Make dfs for analysis

np.clip is for including underflow events into the first bin and overflow events into the last bin

In [ ]:
obvious_cosmic_mask = slc_df.slc.cut.obvious_cosmic
t0_mask = slc_df.slc.cut.t0
is_inside_FV_mask = slc_df.slc.cut.inside_FV
nu_score_mask = slc_df.slc.nu_score > CTE.min_nu_score
track_mask = slc_df.slc.cut.track
shower_mask = slc_df.slc.cut.shower 
chi2_mask = slc_df.slc.cut.MIP_candidates 
angle_mask = slc_df.slc.cut.angle 
proton_BDT_mask = slc_df.slc.cut.proton_BDT
containment_mask = slc_df.slc.cut.containment 
michel_mask = slc_df.slc.cut.michel 
extra_pion_mask = slc_df.slc.cut.extra_pion 
energy_mask = slc_df.slc.cut.energy

# 1. Define the order of cuts
cut_sequence = [
    ("cosmic_rejection", obvious_cosmic_mask & t0_mask & is_inside_FV_mask),
    ("nu_score", nu_score_mask),
    ("track", track_mask),
    ("chi2", chi2_mask),
    ("shower", shower_mask),
    ("angle", angle_mask),
    ("proton_BDT", proton_BDT_mask),
    ("containment", containment_mask),
    ("MIP_refinement", michel_mask & extra_pion_mask),
    ("energy", energy_mask)
]

# 2. Build the cumulative masks
cumulative_masks = {}
current_mask = None

for name, mask in cut_sequence:
    if current_mask is None:
        current_mask = mask
    else:
        current_mask = current_mask & mask
    cumulative_masks[name] = current_mask

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_efficiency(
    var_config,
    var_truth_signal,
    weight_truth_signal,
    var_signal_sel_truth_vec,
    weight_true_sel_signal_vec
):
    # --- Figure with extra width for legend ---
    fig, ax = plt.subplots(figsize=(12, 7))  # wider plot
    
    # --- Denominator histogram (True Signal) ---
    nevts_signal_truth, bins_edges, _ = ax.hist(
        var_truth_signal,
        bins=var_config.bins,
        weights=weight_truth_signal,
        histtype="step",
        label="True Signal",
        color="black",
        linewidth=2
    )
    
    # --- Candidate slices ---
    for i, name in enumerate(cumulative_masks):
        nevts_signal_sel_truth, _, _ = ax.hist(
            var_signal_sel_truth_vec[name],
            bins=var_config.bins,
            weights=weight_true_sel_signal_vec[name],
            histtype="step",
            label=cut_name_nice_map.get(name, name),
            color=COLORS[i % len(COLORS)],
            linewidth=2
        )
    
    # --- Axis formatting ---
    ax.set_ylabel("Events", fontsize=16)
    ax.set_xlabel(var_config.var_plot_name + var_config.var_unit, fontsize=16)
    ax.set_xlim(var_config.bins[0], var_config.bins[-1])
    ax.tick_params(axis='both', labelsize=16)
    ax.margins(x=0)
    
    # --- Legend outside ---
    leg = ax.legend(
        bbox_to_anchor=(1.05, 1),
        loc='upper left',
        borderaxespad=0.,
        fontsize=15,
        framealpha=1.0,
        edgecolor='black',
        fancybox=False
    )
    
    #plt.tight_layout()
    
    # --- Save with extra space for legend ---
    '''
    if save_fig:
        fig.savefig(
            f"{save_fig_dir}/{var_config.var_save_name}-sel_event_rates.pdf",
            bbox_inches='tight',
            bbox_extra_artists=[leg]
        )
    '''
    
    plt.show()
    return fig

In [ ]:
# total generated, for efficiency vector
var_config_vec = {
    VariableConfig.muon_momentum(),
    VariableConfig.muon_direction(), 
    VariableConfig.pion_momentum(), 
    VariableConfig.pion_direction(), 
    VariableConfig.angle_between_candidates(), 
    VariableConfig.num_protons(), 
    VariableConfig.delta_pt(),
    VariableConfig.delta_alpha_T(),
    VariableConfig.delta_phi_T(),
}

for config in var_config_vec:  
    eps = 1e-8
    # Signal event's true muon momentum without event selection
    var_truth_signal = slc_df[slc_df.truth.nu_categ == "CC1pi"][config.var_nu_col]
    var_truth_signal = np.clip(var_truth_signal, config.bins[0], config.bins[-1] - eps)
    weight_truth_signal = slc_df.loc[(slc_df.truth.nu_categ == "CC1pi"), ('slc','wgt','','','','')]
    
    # Total MC reco muon momentum: for fake data
    weight_true_sel_signal_vec = {}
    var_signal_sel_truth_vec = {}
    
    eps = 1e-8
    for name in cumulative_masks:
        # Signal event's true muon momentum after the event selection
        var_signal_sel_truth_vec[name] = slc_df[(slc_df.truth.nu_categ == "CC1pi") & (cumulative_masks[name])][config.var_evt_truth_col]
        var_signal_sel_truth_vec[name] = np.clip(var_signal_sel_truth_vec[name], config.bins[0], config.bins[-1] - eps)
        weight_true_sel_signal_vec[name] = slc_df.loc[(slc_df.truth.nu_categ == "CC1pi") & (cumulative_masks[name]), ('slc','wgt','','','','')]
    
    fig = plot_efficiency(
        config,
        var_truth_signal,
        weight_truth_signal,
        var_signal_sel_truth_vec,
        weight_true_sel_signal_vec
    )


# Response Matrix

Draw true (before event selection) and reco (after event selection) muon momentum distributions of signal events.
Print entries for double check.

In [ ]:
import statsmodels.api as sm
def get_eff_err(success,total):  # success/total
    err = [[],[]]
    eff = success/total
    for i in range(len(success)):
        this_success = success[i]
        this_tot = total[i]
        interval = sm.stats.proportion_confint(this_success,this_tot,method='wilson')
        err[0].append(abs(eff[i]-interval[0]))
        err[1].append(abs(eff[i]-interval[1]))
    return err

In [ ]:

from os import path, makedirs
save_result = True
save_fig = save_result
save_fig_dir = "/exp/sbnd/data/users/lpelegri/Graphs/Efficiency"

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


for config in var_config_vec:  
    eps = 1e-8
    
    var_truth_signal = slc_df[slc_df.truth.nu_categ == "CC1pi"][config.var_nu_col]
    var_truth_signal = np.clip(var_truth_signal, config.bins[0], config.bins[-1] - eps)
    weight_truth_signal = slc_df.loc[(slc_df.truth.nu_categ == "CC1pi"), ('slc','wgt','','','','')]
    
    # Total MC reco muon momentum: for fake data
    weight_true_sel_signal_vec = {}
    var_signal_sel_truth_vec = {}
    
    eps = 1e-8
    for name in cumulative_masks:
        # Signal event's true muon momentum after the event selection
        var_signal_sel_truth_vec[name] = slc_df[(slc_df.truth.nu_categ == "CC1pi") & (cumulative_masks[name])][config.var_evt_truth_col]
        var_signal_sel_truth_vec[name] = np.clip(var_signal_sel_truth_vec[name], config.bins[0], config.bins[-1] - eps)
        weight_true_sel_signal_vec[name] = slc_df.loc[(slc_df.truth.nu_categ == "CC1pi") & (cumulative_masks[name]), ('slc','wgt','','','','')]
    

    fig, ax = plt.subplots(figsize=(10, 8))
        
    # --- 1. Define Binning ---
    bin_edges = config.bins
    if not np.iterable(bin_edges):
        bin_edges = np.linspace(np.min(var_truth_signal), np.max(var_truth_signal), config.bins + 1)
        
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    
    # --- 2. Denominator (Total Signal) ---
    nevts_signal_truth, _ = np.histogram(
        var_truth_signal,
        bins=bin_edges,
        weights=weight_truth_signal
    )
    
    for i, name in enumerate(cumulative_masks):
        # --- 3. Numerator (Selected Signal) ---
        nevts_signal_sel_truth, _ = np.histogram(
            var_signal_sel_truth_vec[name],
            bins=bin_edges,
            weights=weight_true_sel_signal_vec[name]
        )
    
        # --- 4. Efficiency Calculation with Safety ---
        with np.errstate(divide='ignore', invalid='ignore'):
            efficiency = np.divide(
                nevts_signal_sel_truth,
                nevts_signal_truth,
                out=np.zeros_like(nevts_signal_sel_truth, dtype=float),
                where=nevts_signal_truth != 0
            )
    
        # Efficiency errors (assumes get_eff_err is defined globally)
        eff_err = get_eff_err(nevts_signal_sel_truth, nevts_signal_truth)
    
        nice_label = cut_name_nice_map.get(name, name)
        color = COLORS[i % len(COLORS)]
    
        # --- 5. Plotting (The 'Step' Fix) ---
        y_step = np.append(efficiency, efficiency[-1])
        
        ax.step(
            bin_edges,
            y_step,
            where="post",
            label=nice_label,
            linewidth=3,
            color=color,
            zorder=2
        )
    
        ax.errorbar(
            bin_centers,
            efficiency,
            yerr=eff_err,
            fmt='o',
            markersize=5,
            color=color,
            capsize=3,
            zorder=3,
            linestyle='None'
        )
    
    # --- 6. Formatting & Labels ---
    # Draw the dashed grey line at 1
    ax.axhline(1, color='grey', linestyle='--', linewidth=3, alpha=1, zorder=1)

    ax.set_ylabel("Efficiency", fontsize=16)
    
    xlabel = str(config.var_plot_name + " " + config.var_unit)
    ax.set_xlabel(xlabel, fontsize=16)
    
    ax.set_ylim(0, 1.45)
    ax.set_xlim(bin_edges[0], bin_edges[-1])
    ax.margins(x=0)
    y_ticks = np.arange(0, 1.5, 0.1)
    ax.set_yticks(y_ticks)
    
    ax.grid(True, linestyle='--', alpha=0.6) # Lightened grid to not clash with the y=1 line

    
    # --- 7. Legend and Save ---
    leg = ax.legend(
        bbox_to_anchor=(0.5, 0.98),
        loc='upper center',
        borderaxespad=0.,
        fontsize=17,
        ncol = 2,
        framealpha=1.0,
        edgecolor='black',
        fancybox=False
    )
    
    ax.tick_params(axis='both', labelsize=16)
    plt.tight_layout()
    
    if save_fig:
        save_path = f"{save_fig_dir}/{config.var_save_name}_efficiency.pdf"
        fig.savefig(save_path, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    plt.show()